# momentum + 실적

In [1]:
import pandas as pd

def build_universe_from_csv(
    path: str,
    min_cap: float = 3000,
    top_n_liq: int = 500,
    verbose: bool = True
) -> pd.DataFrame:
    """
    csv 기반 유니버스 생성 함수

    Parameters
    ----------
    path : str
        CSV 파일 경로
    min_cap : float
        시가총액 하한 (억 단위)
    top_n_liq : int
        거래대금 상위 N개 (5일평균 기준)
    verbose : bool
        필터링 전/후 개수 출력 여부
    """
    # 1. 데이터 로드
    df = pd.read_csv(path)

    # 2. 시가총액 필터 (유동성/규모)
    cond_cap = df['시가총액(억)'] >= min_cap

    # 3. 거래대금 상위 N개 (5일평균 기준)
    thresh_liq = df['거래대금 (5일평균 억)'].nlargest(top_n_liq).min()
    cond_liq = df['거래대금 (5일평균 억)'] >= thresh_liq

    # 4. 안전성 필터 (관리종목, 스팩, 리츠, 적자기업 등)
    cond_safe = (
        (df['관리종목 =1'] != 1) &
        (df['스팩 =1'] != 1) &
        (df['리츠 =1'] != 1) &
        (df['적자기업 =1'] != 1)
    )

    # 5. 최종 유니버스
    universe = df[cond_cap & cond_liq & cond_safe].copy()

    if verbose:
        print(f"필터링 전: {len(df)}개 -> 필터링 후: {len(universe)}개")
    
    return universe


In [5]:
import pandas as pd
import os

# git 기준 
# 현재 경로 가져오기

current_path = os.getcwd()
print(current_path)

path = current_path
# csv_path = path + '/data/퀀트데이터_260102.csv'
csv_path = path + '/퀀트데이터_260102.csv'

# 데이터 로드
df = pd.read_csv(csv_path)
df.head(1)

c:\Users\cloud\Desktop\PycharmProjects\ai-quant-class\AI퀀트 과정 1기\Day85_project2


,코드,회사명,시장구분,업종대,업종소,주가(원),시가총액(억),시가총액(억) 우선주포함,주식수 (단위:만주),유통주식 비중 (%),...,EPS 24년1Q,EPS 24년2Q,EPS 24년3Q,EPS 24년4Q,EPS 25년1Q,EPS 25년2Q,EPS 25년3Q(E),25년3Q 매출액,25년3Q 영업이익,25년3Q 지배순이익
0,A005930,삼성전자,코스피,반도체 관련장비 및 부품,종합 반도체,128500,7606735,8377015,591964,75.7,...,1109.09,1615.24,1638.51,1269.08,1356.23,833.5,2028.24,860617.47,121660.62,120064.61


In [38]:
universe = build_universe_from_csv(
                path = csv_path, 
                min_cap = 2000,
                top_n_liq = 1500
            )

필터링 전: 2700개 -> 필터링 후: 646개


In [39]:
universe_is_mom = universe.copy()

# 실적 score

In [40]:
# 실적 z-score 계산 (universe 기준)
import numpy as np

col_3q_sales_yoy = '매출액 25년3Q(E) YOY'  # 높을수록 좋음
col_3q_op_yoy = '영업이익 25년3Q(E) YOY'   # 높을수록 좋음
col_3q_ni_yoy = '지배순이익 25년3Q(E) YOY'# 높을수록 좋음


# 1. 성장률 → z-score
for col, new_col in [
    (col_3q_sales_yoy, 'z_sales'),
    (col_3q_op_yoy, 'z_op'),
    (col_3q_ni_yoy, 'z_ni'),
]:
    mu = universe_is_mom[col].mean()
    sigma = universe_is_mom[col].std(ddof=1)
    universe_is_mom[new_col] = (universe_is_mom[col] - mu) / sigma

# 2. 증가분 계산
universe_is_mom['매축액_증가분'] = universe_is_mom['매출액 25년3Q(E)'] - universe_is_mom['매출액 24년3Q']
universe_is_mom['영업이익_증가분'] = universe_is_mom['영업이익 25년3Q(E)'] - universe_is_mom['영업이익 24년3Q']
universe_is_mom['지배순이익_증가분'] = universe_is_mom['지배순이익 25년3Q(E)'] - universe_is_mom['지배순이익 24년3Q']

# 3. 증가분 → z-score
for col, new_col in [
    ('매축액_증가분', 'z_sales_amt'),
    ('영업이익_증가분', 'z_op_amt'),
    ('지배순이익_증가분', 'z_ni_amt'),
]:
    mu = universe_is_mom[col].mean()
    sigma = universe_is_mom[col].std(ddof=1)
    universe_is_mom[new_col] = (universe_is_mom[col] - mu) / sigma

# 4. 실적 종합 z-score
universe_is_mom['z_score_growth'] = (
    (universe_is_mom['z_sales'] * 0.2) +
    (universe_is_mom['z_op'] * 0.1) +
    (universe_is_mom['z_sales_amt'] * 0.4) +
    (universe_is_mom['z_op_amt'] * 0.3)
)

print(universe_is_mom[['회사명', 'z_score_growth']].head())


        회사명  z_score_growth
0      삼성전자        6.981997
1    SK하이닉스        8.558221
3  삼성바이오로직스        0.632930
4       현대차        0.701289
5   HD현대중공업        0.739234


# 모멘텀 score

In [41]:
# 1) 각 기간별 수익률 → z-score (값이 클수록 우수하다고 가정)
mom_cols = [
    ('1개월 등락률 (%)',  'z_mom_1m'),
    ('3개월 등락률 (%)',  'z_mom_3m'),
    ('6개월 등락률 (%)',  'z_mom_6m'),
    ('9개월 등락률 (%)',  'z_mom_9m'),
    ('1년 등락률 (%)',   'z_mom_12m'),
]

for col, zcol in mom_cols:
    mu = universe_is_mom[col].mean()
    sigma = universe_is_mom[col].std(ddof=1)
    universe_is_mom[zcol] = (universe_is_mom[col] - mu) / sigma

# 2) 모멘텀 z-score 종합 (예: 동일 가중 20%씩)
universe_is_mom['z_score_mom'] = (
    0.2 * universe_is_mom['z_mom_1m'] +
    0.2 * universe_is_mom['z_mom_3m'] +
    0.2 * universe_is_mom['z_mom_6m'] +
    0.2 * universe_is_mom['z_mom_9m'] +
    0.2 * universe_is_mom['z_mom_12m']
)

# 통합 스코어 (5:5)

In [44]:
# 6. 통합 스코어 (실적 50% + 모멘텀 50%)
universe_is_mom['z_score_total'] = (
    0.5 * universe_is_mom['z_score_growth'] +
    0.5 * universe_is_mom['z_score_mom']
)

# 7. 상위 100개
final_top = universe_is_mom.sort_values('z_score_total', ascending=False).head(100).copy()
final_top['통합순위'] = range(1, len(final_top) + 1)

final_top[['코드', '회사명', '통합순위',
           'z_score_total', 'z_score_growth', 'z_score_mom']].head(20)

,코드,회사명,통합순위,z_score_total,z_score_growth,z_score_mom
1,A000660,SK하이닉스,1,5.288040,8.558221,2.017859
450,A049630,재영솔루텍,2,4.210862,0.044727,8.376997
236,A458870,씨어스테크놀로지,3,4.200427,2.939409,5.461445
0,A005930,삼성전자,4,4.047630,6.981997,1.113263
14,A055550,신한지주,5,3.646625,7.355275,-0.062026
158,A319400,현대무벡스,6,2.901529,-0.102416,5.905474
120,A108490,로보티즈,7,2.432679,-0.070303,4.935661
6,A402340,SK스퀘어,8,2.412947,2.459713,2.366182
90,A000880,한화,9,2.008020,3.865020,0.151019
345,A476060,온코닉테라퓨틱스,10,1.910808,1.676995,2.144621


In [45]:
# 음수 보정 (절대 모멘텀 필터)
# 1년 수익률이 마이너스인 종목은 추세가 꺾인 것으로 보고 제외
final_df_is_mom = final_top[universe_is_mom['1년 등락률 (%)'] > 0].copy()

C:\Users\cloud\AppData\Local\Temp\ipykernel_21316\4134696084.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  final_df_is_mom = final_top[universe_is_mom['1년 등락률 (%)'] > 0].copy()


In [47]:
from tabulate import tabulate

# 결과 출력
m_cols = ['코드', '회사명', '통합순위', 'z_score_total', 'z_score_growth', 'z_score_mom', '1개월 등락률 (%)', '3개월 등락률 (%)', '6개월 등락률 (%)', '9개월 등락률 (%)', '1년 등락률 (%)']

# final_df에서 Total_Score를 기준으로 정렬하여 상위 20개 출력
print_df = final_df_is_mom.sort_values('z_score_total', ascending=False).head(100)

print(tabulate(print_df[m_cols], headers='keys', tablefmt='psql', floatfmt=".2f"))

+-----+---------+------------------------+------------+-----------------+------------------+---------------+--------------------+--------------------+--------------------+--------------------+------------------+
|     | 코드    | 회사명                 |   통합순위 |   z_score_total |   z_score_growth |   z_score_mom |   1개월 등락률 (%) |   3개월 등락률 (%) |   6개월 등락률 (%) |   9개월 등락률 (%) |   1년 등락률 (%) |
|-----+---------+------------------------+------------+-----------------+------------------+---------------+--------------------+--------------------+--------------------+--------------------+------------------|
|   1 | A000660 | SK하이닉스             |          1 |            5.29 |             8.56 |          2.02 |              21.33 |              71.18 |             142.65 |             271.57 |           295.44 |
| 450 | A049630 | 재영솔루텍             |          2 |            4.21 |             0.04 |          8.38 |             131.16 |             488.17 |             471.26 |             639.58 |  

In [24]:
# CSV 저장
final_df_is_mom[m_cols].to_csv('momentum+earnings_260104.csv', index=False, encoding='utf-8-sig')